# Data for error analysis

Notebook này dùng để chuẩn bị data mẫu cho công đoạn error analysis. Sử dụng dữ liệu M5 (Walmart sales) và thiết kế các biến giải thích (covariates) cho bài toán dự báo chuỗi thời gian. Dữ liệu M5 bao gồm 3049 sản phẩm bán tại 10 cửa hàng của Walmart trong 5 năm. Dữ liệu có cấu trúc phân cấp theo sản phẩm, cửa hàng, bang, bộ phận, và ngành hàng; ngoài ra còn có các biến giải thích như giá bán, khuyến mãi và lịch các sự kiện.

Output là 1 bảng dạng wide với cột `date`, 7 cột `sales` tương ứng với 7 `dept_id`, và các biến đặc trưng sử dụng chung giữa các series.

In [ ]:
# Tải và giải nén data
import os
import zipfile
import urllib.request

ZIP_URL = "https://github.com/Nixtla/m5-forecasts/raw/main/datasets/m5.zip"
ZIP_FILE = "m5.zip"  
EXTRACTED_FILES = [
    "sales_train_evaluation.csv",
    "calendar.csv",
    "sell_prices.csv"
]

def download_and_extract():
    print("Đang tải zip dữ liệu M5")
    urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)
    print("Tải xong, giải nén")
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        for fname in EXTRACTED_FILES:
            z.extract(fname)  
    print("Giải nén xong")

# Kiểm tra 
missing = [f for f in EXTRACTED_FILES if not os.path.exists(f)]
if missing:
    try:
        download_and_extract()
    except Exception as e:
        raise RuntimeError(
            "Không thể tự động tải/giải nén dữ liệu"
            "Tải file m5.zip thủ công, giải nén và đặt các CSV vào thư mục"
            + ZIP_URL
        )
else:
    print("Các file CSV đã tồn tại")

## Load & Prepare

In [14]:
import pandas as pd

# Load data
sales_df = pd.read_csv('sales_train_evaluation.csv')
calendar_df = pd.read_csv('calendar.csv')
sell_prices_df = pd.read_csv('sell_prices.csv')

# Melt to long format
id_cols = ['item_id','dept_id','cat_id','store_id','state_id']
value_cols = [c for c in sales_df.columns if c.startswith('d_')]

sales_long = sales_df.melt(
    id_vars=id_cols, 
    value_vars=value_cols, 
    var_name='d', 
    value_name='sales'
)

# Map d_1 -> date
start_date = pd.to_datetime('2011-01-29')
sales_long['date'] = pd.to_datetime(start_date) + pd.to_timedelta(
    sales_long['d'].str[2:].astype(int) - 1, unit='D'
)

## Merge với Calendar & Sell Prices

In [15]:
# Filter only California
sales_long = sales_long[sales_long['state_id'] == 'CA']

calendar_df['date'] = pd.to_datetime(calendar_df['date'])
calendar_cols = ['date', 'wm_yr_wk', 'event_name_1', 'snap_CA']

merged = sales_long.merge(calendar_df[calendar_cols], on='date', how='left')

merged = merged.merge(
    sell_prices_df[['store_id','item_id','wm_yr_wk','sell_price']],
    on=['store_id','item_id','wm_yr_wk'],
    how='left'
)

# Giữ 1000 ngày cuối
merged = merged[merged['date'] >= merged['date'].max() - pd.Timedelta(days=999)]
merged.reset_index(drop=True, inplace=True)

## Feature Engineering

In [25]:
# Future covariates
merged['month'] = merged['date'].dt.month
merged['weekofyear'] = merged['date'].dt.isocalendar().week.astype(int)
merged['day_of_week'] = merged['date'].dt.dayofweek
merged['is_weekend'] = merged['day_of_week'].isin([5,6]).astype(int)
merged['has_event'] = merged['event_name_1'].notnull().astype(int)

## Aggregate lên Dept-Level

In [26]:
agg_dict = {
    'sales': 'sum',
    'has_event': 'max',
    'snap_CA': 'max',
    'is_weekend': 'max'
}
dept_agg = merged.groupby(['date', 'cat_id'], as_index=False).agg(agg_dict)

## Time Covariates

In [27]:
dept_agg = dept_agg.sort_values(['cat_id', 'date'])

# Thêm lại time covariates (1 dòng mỗi ngày)
time_covs = merged[['date','month','weekofyear','day_of_week']].drop_duplicates()
dept_agg = dept_agg.merge(time_covs, on='date', how='left')

## Pivot sang Wide (Dept-level)

In [28]:
# Pivot sales theo dept_id
tmp = dept_agg.pivot(index='date', columns='cat_id', values='sales')
tmp.columns = [f"{dept}_sales" for dept in tmp.columns]

# Gộp lại với các biến covariates chung
shared_cols = ['date','has_event','snap_CA', 'is_weekend','month','weekofyear','day_of_week']
wide_df = tmp.reset_index().merge(
    dept_agg[shared_cols].drop_duplicates('date'),
    on='date',
    how='left')

In [29]:
wide_df.shape

(1000, 10)

In [30]:
wide_df.to_csv('m5_dept_data.csv', index=False)